# Mutant Moneyball: Tidy Data Cleaning & Exploratory Analysis
The mutant moneyball dataset is a dataset that tracks total comic resale value for different "X-Men" characters across four decades and four different markets.
# Project Overview:
In this project, I receive the excel dataset "mutant moneyball" and apply the "tidy data principles" to it to clean, reshape, and explore visualizations that the dataset outputs through my data tidying-up.

# 1.) Load Libraries

In [ ]:
#load libraries
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
print('matplotlib:', matplotlib.__version__)

# 2.) Load Data
Here, I load the raw data, and see the shape of the data: which is in the "wide format" - which means that each column is made up of two variables; in this case: "decade" and "market."

In [ ]:
#load data
df_untidy = pd.read_csv('mutant_moneyball.csv')
print('Shape:', df_untidy.shape)
print('\nColumn names:')
for col in df_untidy.columns:
    print(' ', col)

df_untidy.head()

# 3.) Missing Data Observation
Here, I look at the types of data and see that the heritage and ebay columns are stored as integer variables, while the wiz and oStreet columns are stored as strings.

Then, I see just how many pieces of data are missing from each column.

In [ ]:
#observe data types and missing data
print('Data types:')
print(df_untidy.dtypes)
print('Missing values per column:')
print(df_untidy.isnull().sum())

# 4.) Data Cleaning and Transformation
Performed through:
1. First melting the data into long format (vs. the original wide) using df_untidy.melt
    - After melting, each row represents one "X-Man," one Decade, and one Market type
2. Splitting the variable column into "decade" and "market" using str.split()
3. Cleaning Decade column and TotalValue column with str.replace()
4. Finalizing output
    - Select the four tidy columns, drop rows where no sale value recorded, and reset index

In [ ]:
#melt
df_melted = df_untidy.melt(id_vars=['Member'], var_name='variable', value_name='TotalValue')
print('Shape after melting:', df_melted.shape)
print('Sample of melted data:')
df_melted.head(10)

#split
split_cols = df_melted['variable'].str.split('_', expand=True)
df_melted['decade_raw'] = split_cols[0]
df_melted['Market'] = split_cols[1]
print('Unique decade_raw values:', df_melted['decade_raw'].unique())
print('Unique Market values:', df_melted['Market'].unique())

#clean
df_melted['Decade'] = df_melted['decade_raw'].str.replace('TotalValue', '', regex=False)
df_melted['TotalValue'] = (
    df_melted['TotalValue']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)

df_melted['TotalValue'] = pd.to_numeric(df_melted['TotalValue'], errors='coerce')

#finalize output
df_tidy = (
    df_melted[['Member', 'Decade', 'Market', 'TotalValue']]
    .dropna(subset=['TotalValue'])
    .reset_index(drop=True)
)

print('Final tidy shape:', df_tidy.shape)
print('Example of first couple of rows of tidy data:')
df_tidy.head(10)

# Visualizations: Pivot Table
Table derived from total card value by "Decade" and "Market," seeing the aggregated data

In [ ]:
#create pivot table
pivot_sum = df_tidy.pivot_table(
    values='TotalValue',
    index='Decade',
    columns='Market',
    aggfunc='sum'
)
pivot_sum = pivot_sum.reindex(['60s', '70s', '80s', '90s'])
print('Total Card Value by Decade and Market:')
pivot_sum.style.format('${:,.2f}')

# Visualizations: Horizontal Bar Chart
This chart ranks the top 10 "X-Men" members by combined card value across all decades and markets.

In [ ]:
#create horizontal bar chart
fig, ax = plt.subplots(figsize=(10, 6))

top10 = (
    df_tidy.groupby('Member')['TotalValue']
    .sum()
    .sort_values(ascending=True)
    .tail(10)
)

colors = plt.cm.viridis(np.linspace(0.2, 0.85, len(top10)))
bars = ax.barh(top10.index, top10.values)

#put value labels at the end of each bar
for bar, val in zip(bars, top10.values):
    ax.text(
        bar.get_width() + max(top10.values) * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f'${val:,.0f}', va='center', ha='left', fontsize=9
    )

ax.set_title('Top 10 X-Men by Total Card Value (All Decades and Markets)')
ax.set_xlabel('Total Card Value (USD)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.grid(axis='x')
plt.show()

# Visualizations: Grouped Bar Chart by Decade
This chart compares total card sales across all four markets from each decade - showing how the collectibles market diminished over time

In [ ]:
#create grouped bar chart
fig, ax = plt.subplots(figsize=(12, 6))

grouped = (
    df_tidy.groupby(['Decade', 'Market'])['TotalValue']
    .sum()
    .unstack()
    .reindex(['60s', '70s', '80s', '90s'])
)

grouped.plot(kind='bar', ax=ax)
ax.set_title('Total X-Men Card Value by Decade and Market')
ax.set_xlabel('Decade')
ax.set_ylabel('Total Card Value (USD)')
ax.set_xticklabels(['60s', '70s', '80s', '90s'], rotation=0)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(title='Market',  loc='upper right')
ax.grid(axis='y')
plt.show()

# Summary
Actions I performed to the "Mutant Moneyball" dataset to tidy it's data:

1.) Loaded the raw Mutant Moneyball dataset (26 members × 16 value columns in wide format)

2.) Identified untidy structure: "Decade" and "Market" were encoded in column names, not stored as values. The "wiz" and "oStreet" columns also had dollar signs and commas requiring cleanup

3.) Melted the data with pd.melt() to convert 16 columns into a long-format

4.) Split the Variable column with str.split() to make two new columns

5.) Cleaned labels and values with str.replace()

6.) Derived pivot tables from total card value by "Decade" and "Market," seeing the aggregated data